# DBSCAN — Regionais de Saúde de MG (painel Regional x Ano)

Unidade de agrupamento: par **(REGIONAL DE SAÚDE, ANO)**. Uma mesma Regional pode cair em agrupamentos diferentes em anos diferentes -- isso é informação (mudança de perfil epidemiológico ao longo do tempo), não inconsistência.

`eps` é estimado pelo método de Monko & Kimura (2023) -- distância média aos k vizinhos, amostragem estratificada, ajuste polinomial e joelho (kneed). `MinPts` vem de um grid search com `eps` fixo, escolhido pela maior silhueta. Nenhum dos dois é fixado à mão.

Toda a lógica reutilizável está em `utilitarios_agrupamentoManual.py` (pipeline) e `graficos_agrupamentoManual.py` (visualizações); este notebook só configura e executa. O diagnóstico de sensibilidade a parâmetros (PCA, `K_VIZINHOS_EPS`) foi movido para `diagnostico_parametrosManual.ipynb`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd()))


from utilitarios_agrupamentoManual import (
    COLUNA_ANO, COLUNA_REGIONAL, ROTULO_RUIDO,
    BLOCOS_DE_TAXA, PASTA_TAXA,
    K_VIZINHOS_EPS, QUANTIDADE_DE_ESTRATOS, AMOSTRAS_POR_ESTRATO,
    GRAU_POLINOMIO_EPS, SENSIBILIDADE_JOELHO,
    MIN_SAMPLES_MINIMO, MIN_SAMPLES_PASSO,
    USAR_PCA, RAZAO_MAXIMA_ATRIBUTOS_POR_LINHA, VARIANCIA_EXPLICADA_ALVO,
    NUMERO_MAXIMO_DE_COMPONENTES, METRICA_DISTANCIA, SEMENTE_ALEATORIA,
    LIMIAR_MAXIMO_DE_NULOS_POR_COLUNA,
    USAR_JANELA_COMUM, ANO_INICIAL_COMUM, ANO_FINAL_COMUM,
    montar_painel, calcular_janela_comum, restringir_a_janela_comum,
    descartar_colunas_sem_informacao_e_total, descartar_colunas_com_muitos_nulos, tratar_nulos_sem_imputar,
    padronizar_e_reduzir, preparar_projecao_2d, pesos_das_variaveis_no_componente,
    estimar_eps_monko_kimura, buscar_min_samples,
    executar_dbscan, resumir_agrupamento,
    montar_resumo_parametros, montar_tabela_pertencimento, montar_visao_agregada_regional_ano,
    montar_resumo_por_cluster,
    registrar_log,
)
from graficos_agrupamentoManual import (
    grafico_variancia_explicada, grafico_pesos_da_componente,
    grafico_k_distancias, grafico_silhueta_por_min_samples, grafico_dispersao_pc1_pc2,
    grafico_dispersao_pc1_pc2_por_ano,
    grafico_distribuicao_por_regional, grafico_distribuicao_por_ano,
    grafico_heatmap_regional_ano,
)

## 1. Leitura dos CSVs de taxa e reformatação (largo -> painel)

In [2]:
painel = montar_painel(blocos=BLOCOS_DE_TAXA, pasta_taxa=PASTA_TAXA)
painel.head()

[LOG] tabela_casos_por_faixa_etária_taxa.csv: 9 categoria(s), 1064 par(es) regional-ano.
[LOG] tabela_casos_por_escolaridade_taxa.csv: 8 categoria(s), 728 par(es) regional-ano.
[LOG] tabela_casos_por_raça_e_cor_taxa.csv: 7 categoria(s), 728 par(es) regional-ano.
[LOG] tabela_mortes_por_faixa_etária_taxa.csv: 9 categoria(s), 1288 par(es) regional-ano.
[LOG] tabela_mamografias_por_faixa_etária_taxa.csv: 9 categoria(s), 364 par(es) regional-ano.
[LOG] Painel montado: 1316 par(es) regional-ano x 42 variável(is), 28 regional(is).


casos-faixa_0-9  casos-faixa_10-19  casos-faixa_20-29  \
REGIONAL DE SAÚDE ANO                                                           
alfenas           1979              NaN                NaN                NaN   
                  1980              NaN                NaN                NaN   
                  1981              NaN                NaN                NaN   
                  1982              NaN                NaN                NaN   
                  1983              NaN                NaN                NaN   

                        casos-faixa_30-39  casos-faixa_40-49  \
REGIONAL DE SAÚDE ANO                                          
alfenas           1979                NaN                NaN   
                  1980                NaN                NaN   
                  1981                NaN                NaN   
                  1982                NaN                NaN   
                  1983                NaN                NaN   

                        casos-faixa_50-59  casos-faixa_60-69  \
REGIONAL DE SAÚDE ANO                                          
alfenas           1979                NaN                NaN   
                  1980                NaN                NaN   
                  1981                NaN                NaN   
                  1982                NaN                NaN   
                  1983                NaN                NaN   

                        casos-faixa_70-ou-mais  casos-faixa_total  \
REGIONAL DE SAÚDE ANO                                               
alfenas           1979                     NaN                NaN   
                  1980                     NaN                NaN   
                  1981                     NaN                NaN   
                  1982                     NaN                NaN   
                  1983                     NaN                NaN   

                        casos-escolaridade_fundamental-completo  ...  \
REGIONAL DE SAÚDE ANO                                            ...   
alfenas           1979                                      NaN  ...   
                  1980                                      NaN  ...   
                  1981                                      NaN  ...   
                  1982                                      NaN  ...   
                  1983                                      NaN  ...   

                        mortalidade_total  mamografia_0-9  mamografia_10-19  \
REGIONAL DE SAÚDE ANO                                                         
alfenas           1979               3.54             NaN               NaN   
                  1980               7.47             NaN               NaN   
                  1981               3.99             NaN               NaN   
                  1982               3.38             NaN               NaN   
                  1983               7.82             NaN               NaN   

                        mamografia_20-29  mamografia_30-39  mamografia_40-49  \
REGIONAL DE SAÚDE ANO                                                          
alfenas           1979               NaN               NaN               NaN   
                  1980               NaN               NaN               NaN   
                  1981               NaN               NaN               NaN   
                  1982               NaN               NaN               NaN   
                  1983               NaN               NaN               NaN   

                        mamografia_50-59  mamografia_60-69  \
REGIONAL DE SAÚDE ANO                                        
alfenas           1979               NaN               NaN   
                  1980               NaN               NaN   
                  1981               NaN               NaN   
                  1982               NaN               NaN   
                  1983               NaN               NaN   

                   

## 2. Janela temporal, descarte de colunas e tratamento de nulos

`USAR_JANELA_COMUM=True` restringe o painel à interseção dos períodos declarados de cada bloco (ou ao período manual `ANO_INICIAL_COMUM`/`ANO_FINAL_COMUM`, se informado); `False` usa o período completo de cada bloco, sem cortar ano nenhum.

Nos CSVs de taxa, uma célula fica em branco (NaN) quando a soma bruta da categoria-ano-regional é NaN em todos os municípios -- `calculo_taxa.ipynb` preserva essa ambiguidade de propósito (`sum(min_count=1)`): não dá para distinguir "zero eventos" de "sem população válida para calcular" só olhando a taxa. Por isso **nenhum nulo vira 0 aqui**. A sequência abaixo só remove o que é estrutural (fora da janela, categoria sem-informação/total, atributo quase todo nulo) e descarta as linhas que ainda restarem com algum valor ausente -- sem preencher nada.

In [3]:
if USAR_JANELA_COMUM:
    janela = calcular_janela_comum(BLOCOS_DE_TAXA, ano_inicial_manual=ANO_INICIAL_COMUM, ano_final_manual=ANO_FINAL_COMUM)
    matriz = restringir_a_janela_comum(painel, janela)
else:
    registrar_log("USAR_JANELA_COMUM=False: usando o período completo de cada bloco, sem restringir ano.")
    matriz = painel

matriz = descartar_colunas_sem_informacao_e_total(matriz)
matriz = descartar_colunas_com_muitos_nulos(matriz, limiar_maximo=LIMIAR_MAXIMO_DE_NULOS_POR_COLUNA)
matriz = tratar_nulos_sem_imputar(matriz)

registrar_log(f"Matriz final: {matriz.shape[0]} linha(s) x {matriz.shape[1]} atributo(s).")
matriz.head()

[LOG] Janela comum: 2013-2023 (definida manualmente).
[LOG] Restrita à janela comum 2013-2023: 308 de 1316 par(es) regional-ano.
[LOG] Descartada(s) 7 coluna(s) sem-informação/total: ['casos-faixa_total', 'casos-escolaridade_sem-informacao', 'casos-escolaridade_total', 'casos-raca-cor_sem-informacao', 'casos-raca-cor_total', 'mortalidade_total', 'mamografia_total']
[LOG] Descartada(s) 3 coluna(s) com mais de 70% de nulos: ['mortalidade_0-9', 'mortalidade_10-19', 'mortalidade_20-29']
[LOG] Descartada(s) 194 de 308 linha(s) (63.0%) por ter ao menos um valor ausente -- nenhum valor foi imputado.
[LOG] Matriz final: 114 linha(s) x 32 atributo(s).


casos-faixa_0-9  casos-faixa_10-19  casos-faixa_20-29  \
REGIONAL DE SAÚDE ANO                                                           
alfenas           2015              0.0                0.0              11.37   
                  2018              0.0                0.0               0.00   
                  2020              0.0                0.0               0.00   
                  2021              0.0                0.0              15.04   
                  2023              0.0                0.0               6.13   

                        casos-faixa_30-39  casos-faixa_40-49  \
REGIONAL DE SAÚDE ANO                                          
alfenas           2015              76.55             150.01   
                  2018              49.72             132.57   
                  2020              86.79              74.51   
                  2021              41.91             103.75   
                  2023              61.31             146.59   

                        casos-faixa_50-59  casos-faixa_60-69  \
REGIONAL DE SAÚDE ANO                                          
alfenas           2015             204.35             177.21   
                  2018             118.94             140.00   
                  2020             118.04             167.39   
                  2021             173.92             157.07   
                  2023             164.66             138.01   

                        casos-faixa_70-ou-mais  \
REGIONAL DE SAÚDE ANO                            
alfenas           2015                  251.93   
                  2018                  147.80   
                  2020                  171.10   
                  2021                  140.78   
                  2023                  204.25   

                        casos-escolaridade_fundamental-completo  \
REGIONAL DE SAÚDE ANO                                             
alfenas           2015                                    23.77   
                  2018                                    34.26   
                  2020                                    41.36   
                  2021                                    41.29   
                  2023                                    29.31   

                        casos-escolaridade_fundamental-incompleto  ...  \
REGIONAL DE SAÚDE ANO                                              ...   
alfenas           2015                                      40.22  ...   
                  2018                                      10.23  ...   
                  2020                                       6.53  ...   
                  2021                                      13.33  ...   
                  2023                                      28.05  ...   

                        mortalidade_60-69  mortalidade_70-ou-mais  \
REGIONAL DE SAÚDE ANO                                               
alfenas           2015              10.13                   50.39   
                  2018              18.06                   41.38   
                  2020              46.03                   67.40   
                  2021              24.17                  106.80   
                  2023              18.65                   59.57   

                        mamografia_0-9  mamografia_10-19  mamografia_20-29  \
REGIONAL DE SAÚDE ANO                                                        
alfenas           2015            0.00              0.00              0.00   
                  2018            0.00             12.74            125.76   
                  2020            7.48             16.73            160.93   
                  2021            7.56              3.43             75.20   
                  2023            0.00              7.20             39.84   

                        mamografia_30-39  mamografia_40-49  mamografia_50-59  \
REGIONAL DE SAÚDE ANO                                                          
alfenas           2015     

## 3. Padronização (RobustScaler) e PCA

`USAR_PCA=None` decide sozinho pela razão atributos/linhas (`RAZAO_MAXIMA_ATRIBUTOS_POR_LINHA`); `True` força PCA, `False` pula o PCA inteiramente -- em ambos os casos o critério automático é ignorado.

In [4]:
matriz_padronizada, matriz_para_dbscan, pca, variancia_acumulada = padronizar_e_reduzir(
    matriz,
    razao_maxima_atributos_por_linha=RAZAO_MAXIMA_ATRIBUTOS_POR_LINHA,
    variancia_alvo=VARIANCIA_EXPLICADA_ALVO,
    numero_maximo_de_componentes=NUMERO_MAXIMO_DE_COMPONENTES,
    semente=SEMENTE_ALEATORIA,
    usar_pca=USAR_PCA,
)

[LOG] PCA não aplicado: USAR_PCA=False (forçado; razão atributos/linhas = 0.281).


### Diagnóstico: variância por componente e pesos das variáveis

Só roda quando o PCA foi de fato aplicado (`pca is not None`). O "scree plot" mostra a variância explicada por componente, com PC1 destacado em vermelho se ultrapassar `VARIANCIA_PC1_ALERTA` -- direção dominante isolada sugere que o componente captura porte/volume da regional, não perfil epidemiológico. Os pesos ("loadings") de PC1 mostram quais atributos originais mais influenciam esse eixo.

In [5]:
if pca is not None:
    grafico_variancia_explicada(pca, variancia_acumulada).show()
    display(pesos_das_variaveis_no_componente(pca, matriz.columns, componente=1))
    grafico_pesos_da_componente(pca, matriz.columns, componente=1).show()
else:
    registrar_log("USAR_PCA resultou em pca=None -- diagnóstico de variância/pesos não se aplica (DBSCAN rodou sobre os atributos originais).")

[LOG] USAR_PCA resultou em pca=None -- diagnóstico de variância/pesos não se aplica (DBSCAN rodou sobre os atributos originais).


## 4. Estimativa de eps (Monko & Kimura, 2023)

In [6]:
resultado_eps = estimar_eps_monko_kimura(
    matriz_para_dbscan,
    k=K_VIZINHOS_EPS, p=QUANTIDADE_DE_ESTRATOS, m=AMOSTRAS_POR_ESTRATO,
    grau_polinomio=GRAU_POLINOMIO_EPS, sensibilidade=SENSIBILIDADE_JOELHO,
    semente=SEMENTE_ALEATORIA, metric=METRICA_DISTANCIA,
)
eps_escolhido = resultado_eps["eps"]

figura_k_distancias = grafico_k_distancias(resultado_eps, k=K_VIZINHOS_EPS)
figura_k_distancias.show()

[LOG] eps (Monko & Kimura, k=5, p=10, m=5, grau=3): 4.7581 no índice 84 de 114.


## 5. Estimativa de MinPts (grid search, eps fixo)

In [7]:
min_samples_escolhido, tabela_grid_min_samples = buscar_min_samples(
    matriz_para_dbscan, eps_escolhido,
    minimo=MIN_SAMPLES_MINIMO, passo=MIN_SAMPLES_PASSO,
    metric=METRICA_DISTANCIA,
)

figura_silhueta = grafico_silhueta_por_min_samples(tabela_grid_min_samples, min_samples_escolhido)
figura_silhueta.show()
tabela_grid_min_samples.head(10)

[LOG] ATENÇÃO: nenhum MinPts da grade produziu >= 2 agrupamentos válidos para a silhueta -- mantendo o menor MinPts testado.
[LOG] MinPts escolhido: 3 (grade 3-113, passo 1).


,min_samples,n_clusters,n_ruido,silhueta,silhueta_com_ruido
0,3,1,26,NaN,0.251151
1,4,1,26,NaN,0.251151
2,5,1,27,NaN,0.248513
3,6,1,27,NaN,0.248513
4,7,1,27,NaN,0.248513
5,8,1,27,NaN,0.248513
6,9,1,27,NaN,0.248513
7,10,1,27,NaN,0.248513
8,11,1,27,NaN,0.248513
9,12,1,27,NaN,0.248513


## 6. DBSCAN

In [8]:
modelo_dbscan = executar_dbscan(matriz_para_dbscan, eps_escolhido, min_samples_escolhido, metric=METRICA_DISTANCIA)
rotulos = modelo_dbscan.labels_

n_clusters, n_ruido, silhueta_sem_ruido, silhueta_com_ruido = resumir_agrupamento(matriz_para_dbscan, rotulos)
registrar_log(f"DBSCAN(eps={eps_escolhido:.4f}, min_samples={min_samples_escolhido}, metric='{METRICA_DISTANCIA}')")
registrar_log(f"  Agrupamentos: {n_clusters}")
registrar_log(f"  Ruído: {n_ruido} de {len(rotulos)} par(es) ({n_ruido / len(rotulos):.1%})")
registrar_log(f"  Silhueta (sem ruído): {silhueta_sem_ruido:.4f}")
registrar_log(f"  Silhueta (ruído como cluster próprio -- métrica distorcida, só para comparação): {silhueta_com_ruido:.4f}")

[LOG] DBSCAN(eps=4.7581, min_samples=3, metric='euclidean')
[LOG]   Agrupamentos: 1
[LOG]   Ruído: 26 de 114 par(es) (22.8%)
[LOG]   Silhueta (sem ruído): nan
[LOG]   Silhueta (ruído como cluster próprio -- métrica distorcida, só para comparação): 0.2512


## Visualizações

In [9]:
matriz_2d, variancia_explicada_2d = preparar_projecao_2d(matriz_padronizada, matriz_para_dbscan, pca, semente=SEMENTE_ALEATORIA)

figura_dispersao = grafico_dispersao_pc1_pc2(
    matriz_2d, rotulos, matriz.index, variancia_explicada_2d, eps_escolhido, min_samples_escolhido,
)
figura_dispersao.show()

### Dispersão PC1 x PC2 por ano

Mesmos pontos e paleta do gráfico acima, mas em uma grade de mini-gráficos -- um por ano -- para comparar a composição dos agrupamentos ano a ano em vez de ver todos os pontos misturados.

In [10]:
figura_dispersao_por_ano = grafico_dispersao_pc1_pc2_por_ano(matriz_2d, rotulos, matriz.index, variancia_explicada_2d)
figura_dispersao_por_ano.show()

In [11]:
resultado = montar_tabela_pertencimento(matriz.index, rotulos)

figura_regional = grafico_distribuicao_por_regional(resultado)
figura_regional.show()

figura_ano = grafico_distribuicao_por_ano(resultado)
figura_ano.show()

### Composição de cada cluster (tabela, não só o gráfico)

A dispersão acima projeta em PC1 x PC2 -- dois clusters que o DBSCAN separou no espaço original podem aparecer sobrepostos no 2D. Esta tabela lista, por CLUSTER (ruído = -1), quantos pares e quais Regional-Ano pertencem a ele, direto dos rótulos do DBSCAN, sem depender da projeção.

In [12]:
resumo_por_cluster = montar_resumo_por_cluster(resultado)
resumo_por_cluster

,CLUSTER,N_PARES,N_REGIONAIS_DISTINTAS,MEMBROS
0,-1,26,13,"alfenas (2018), alfenas (2020), alfenas (2023)..."
1,0,88,25,"alfenas (2015), alfenas (2021), barbacena (202..."


## Saídas finais

In [13]:
resumo_parametros = montar_resumo_parametros(
    K_VIZINHOS_EPS, QUANTIDADE_DE_ESTRATOS, AMOSTRAS_POR_ESTRATO,
    eps_escolhido, min_samples_escolhido, METRICA_DISTANCIA,
    usar_robust_scaler=True, pca=pca, variancia_acumulada=variancia_acumulada,
    n_linhas=matriz.shape[0], n_atributos=matriz.shape[1],
    n_clusters=n_clusters, n_ruido=n_ruido,
    silhueta_sem_ruido=silhueta_sem_ruido, silhueta_com_ruido=silhueta_com_ruido,
)
resumo_parametros

,K_VIZINHOS_EPS,QUANTIDADE_DE_ESTRATOS,AMOSTRAS_POR_ESTRATO,EPS,MIN_SAMPLES,METRICA_DISTANCIA,ESCALONADOR,PCA_APLICADO,PCA_N_COMPONENTES,PCA_VARIANCIA_EXPLICADA,N_LINHAS,N_ATRIBUTOS,N_CLUSTERS,N_RUIDO,SILHUETA_SEM_RUIDO,SILHUETA_COM_RUIDO
0,5,10,5,4.758099,3,euclidean,RobustScaler,False,NaN,NaN,114,32,1,26,NaN,0.251151


In [14]:
atributos_usados = pd.DataFrame({"ATRIBUTO": list(matriz.columns)})
atributos_usados

,ATRIBUTO
0,casos-faixa_0-9
1,casos-faixa_10-19
2,casos-faixa_20-29
3,casos-faixa_30-39
4,casos-faixa_40-49
5,casos-faixa_50-59
6,casos-faixa_60-69
7,casos-faixa_70-ou-mais
8,casos-escolaridade_fundamental-completo
9,casos-escolaridade_fundamental-incompleto


In [15]:
resultado

,REGIONAL DE SAÚDE,ANO,CLUSTER
0,alfenas,2015,0
1,alfenas,2018,-1
2,alfenas,2020,-1
3,alfenas,2021,0
4,alfenas,2023,-1
...,...,...,...
109,varginha,2015,0
110,varginha,2018,0
111,varginha,2020,0
112,varginha,2021,0


In [16]:
visao_agregada_regional_ano = montar_visao_agregada_regional_ano(resultado)
visao_agregada_regional_ano

ANO,2015,2018,2020,2021,2023
REGIONAL DE SAÚDE,,,,,
alfenas,0.0,-1.0,-1.0,0.0,-1.0
barbacena,NaN,-1.0,0.0,NaN,NaN
belo horizonte,0.0,0.0,0.0,0.0,0.0
coronel fabriciano,-1.0,-1.0,0.0,0.0,0.0
diamantina,0.0,0.0,NaN,0.0,0.0
divinopolis,0.0,0.0,0.0,0.0,0.0
governador valadares,0.0,0.0,0.0,0.0,-1.0
itabira,0.0,0.0,-1.0,-1.0,-1.0
ituiutaba,0.0,NaN,0.0,0.0,NaN


### Heatmap: cluster por Regional e Ano

Mesma informação de `visao_agregada_regional_ano`, mas como cor em vez de número -- uma cor por CLUSTER (ruído incluído), célula em branco onde não há dado (par descartado no dropna ou fora da cobertura de algum bloco).

In [17]:
figura_heatmap = grafico_heatmap_regional_ano(visao_agregada_regional_ano)
figura_heatmap.show()